# Lab 3：卷积神经网络与 MNIST

本 Notebook 对应实验三。它用一个小型 CNN 完成 MNIST 分类，并把卷积、池化、展平、全连接和训练循环串起来。代码可以直接上传 Colab。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(7)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_set = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_set = Subset(train_set, range(10000))
test_set = Subset(test_set, range(2000))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
print(len(train_set), len(test_set), device)

In [ ]:
images, labels = next(iter(train_loader))
plt.figure(figsize=(8, 2))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(images[i, 0], cmap='gray')
    plt.title(str(labels[i].item()))
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 5, padding=2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
for epoch in range(2):
    model.train()
    total_loss = 0.0
    for x, target in train_loader:
        x, target = x.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print('epoch', epoch + 1, 'mean loss', total_loss / len(train_loader))

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for x, target in test_loader:
        pred = model(x.to(device)).argmax(dim=1).cpu()
        correct += (pred == target).sum().item()
        total += target.numel()
print('test accuracy:', correct / total)

## 形状追踪

输入是 `1×28×28`。第一次池化后为 `16×14×14`，第二次池化后为 `32×7×7`，所以全连接层的输入必须写成 `32*7*7`。这也是实验中出现模型报错时最应该优先检查的地方：上一层输出通道和下一层输入通道、空间尺寸必须一致。